# Prefix-Cache Serving — RadixAttention and KV Reuse: Interactive Visual Explorer

> Treat the KV cache as a first-class, reusable resource stored in a radix tree, and scheduling changes with it: instead of FCFS (first-come, first-served) as vLLM schedules, a cache-aware scheduler prioritizes requests with longer shared prefixes — effectively a depth-first radix traversal so hot branches stay resident in HBM. SGLang is the engine that built serving around this idea. On Llama 3.1 8B with ShareGPT-like 1K prompts, SGLang hits ~16,200 tok/s to vLLM's ~12,500, a ~29% edge. On prefix-heavy RAG workloads the advantage reaches 6.4x. On voice-cloning-shaped workloads cache hit rate cleared 86%. Deployed on 400,000+ GPUs in 2026 across xAI, LinkedIn, Cursor, Oracle, GCP, Azure, AWS. The gotcha is that the 6.4x number evaporates when prefix ordering is inconsistent — ordering is the engineer's lever.

Welcome to the interactive companion notebook for **Prefix-Cache Serving — RadixAttention and KV Reuse**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Toy RadixAttention scheduler — stdlib Python.

Simulate an SGLang-style radix-tree KV cache plus two schedulers:
  FCFS         : naive first-come first-served
  CACHE_AWARE  : depth-first dispatch on hottest branch

Also show how scrambled prompt ordering collapses hit rate. Pedagogical
constants — the shape matches the published numbers, not the absolute
latencies.
"""

from __future__ import annotations


In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
import random

KV_BUDGET_BLOCKS = 160    # small budget so eviction bites under FCFS
BLOCK_TOKENS = 16

def token_count(seg: str) -> int:
    if seg == "SYSTEM":
        return 2000
    if seg.startswith("DOC_"):
        return 500
    if seg.startswith("Q_"):
        return 60
    if seg == "TOOLS":
        return 300
    return 100


In [ ]:
@dataclass
class Request:
    rid: int
    segments: list[str]

class RadixCache:
    """Represent the tree as a dict: path_tuple -> blocks (last_used)."""

def __init__(self, budget_blocks: int = KV_BUDGET_BLOCKS):
        self.budget = budget_blocks
        self.used = 0
        self.time = 0
        # key: tuple of segments. value: (blocks, last_used)
        self.nodes: dict[tuple[str, ...], list[int]] = {}


In [ ]:
def walk(self, segments: list[str]) -> int:
        """Return number of tokens that are already cached at the longest matching
        prefix, bumping last_used along the path."""
        reused = 0
        self.time += 1
        for i in range(1, len(segments) + 1):
            key = tuple(segments[:i])
            if key in self.nodes:
                reused += token_count(segments[i - 1])
                self.nodes[key][1] = self.time
            else:
                break
        return reused


In [ ]:
def insert(self, segments: list[str]) -> None:
        """Insert any missing segments on the path, evicting LRU leaves if over budget."""
        for i in range(1, len(segments) + 1):
            key = tuple(segments[:i])
            if key in self.nodes:
                continue
            blocks = (token_count(segments[i - 1]) + BLOCK_TOKENS - 1) // BLOCK_TOKENS
            while self.used + blocks > self.budget and self._evict_one():
                pass
            self.nodes[key] = [blocks, self.time]
            self.used += blocks


In [ ]:
def _evict_one(self) -> bool:
        leaves = [k for k in self.nodes if not any(
            other != k and other[: len(k)] == k for other in self.nodes)]
        if not leaves:
            return False
        victim = min(leaves, key=lambda k: self.nodes[k][1])
        self.used -= self.nodes.pop(victim)[0]
        return True

def simulate(requests: list[Request], scheduler: str) -> dict:
    cache = RadixCache()


In [ ]:
if scheduler == "CACHE_AWARE":
        branch_count: dict[tuple[str, ...], int] = defaultdict(int)
        for r in requests:
            for i in range(1, len(r.segments) + 1):
                branch_count[tuple(r.segments[:i])] += 1

def score(r: Request) -> int:
            return max(branch_count[tuple(r.segments[:i])] * sum(
                token_count(s) for s in r.segments[:i])
                for i in range(1, len(r.segments) + 1))
        order = sorted(requests, key=score, reverse=True)
    else:
        order = list(requests)


In [ ]:
saved = 0
    total = 0
    for r in order:
        prompt_tokens = sum(token_count(s) for s in r.segments)
        total += prompt_tokens
        reused = cache.walk(r.segments)
        saved += reused
        cache.insert(r.segments)

return {
        "hit_rate": saved / total if total else 0,
        "saved": saved,
        "total": total,
        "reqs": len(requests),
    }

def workload_rag(n: int = 80, docs: int = 4, seed: int = 1) -> list[Request]:
    rng = random.Random(seed)
    reqs = []
    for i in range(n):
        doc = f"DOC_{rng.randrange(docs)}"
        q = f"Q_{i}"
        reqs.append(Request(i, ["SYSTEM", "TOOLS", doc, q]))
    rng.shuffle(reqs)
    return reqs


In [ ]:
def workload_scrambled(n: int = 80, docs: int = 4, seed: int = 1) -> list[Request]:
    """Prompts reorder [SYSTEM, TOOLS, DOC] randomly. Tree cannot share the prefix."""
    rng = random.Random(seed)
    reqs = []
    for i in range(n):
        doc = f"DOC_{rng.randrange(docs)}"
        q = f"Q_{i}"
        prefix = ["SYSTEM", "TOOLS", doc]
        rng.shuffle(prefix)
        reqs.append(Request(i, prefix + [q]))
    rng.shuffle(reqs)
    return reqs


In [ ]:
def report(label: str, res: dict) -> None:
    print(f"{label:44}  hit_rate={res['hit_rate']:6.1%}   "
          f"saved={res['saved']:>6}/{res['total']:<6} tok   reqs={res['reqs']}")

def main() -> None:
    print("=" * 88)
    print("TOY RADIX CACHE — cache hit rate across schedulers and orderings")
    print("=" * 88)

rag = workload_rag()
    report("RAG workload | FCFS", simulate(rag, "FCFS"))
    report("RAG workload | CACHE_AWARE", simulate(rag, "CACHE_AWARE"))


In [ ]:
scrambled = workload_scrambled()
    report("RAG scrambled prefix | FCFS", simulate(scrambled, "FCFS"))
    report("RAG scrambled prefix | CACHE_AWARE", simulate(scrambled, "CACHE_AWARE"))

print()
    print("=" * 88)
    print("KEY FINDING")
    print("-" * 88)
    print("  Fixed ordering + cache-aware scheduler : hit rate clears 80% on RAG.")
    print("  Scrambled prefix order : hit rate collapses — the tree cannot find shared paths.")
    print("  Real cases: 7% -> 74% hit rate by moving dynamic content out of the prefix.")


In [ ]:
if __name__ == "__main__":
    main()
